# 3. Model Training & Evaluation
**AAI-540 MLOps Project — Group 6: Student Performance Prediction**

This notebook covers:
- Train/Validation/Test split (70/15/15)
- Training 3 models: Logistic Regression, Decision Tree, Random Forest
- Hyperparameter tuning with GridSearchCV
- Model comparison using Accuracy, Precision, Recall, F1-Score
- Confusion matrices and classification reports
- Feature importance analysis
- Saving the best model for deployment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score, roc_curve)
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 3.1 Load Processed Data

In [ ]:
df = pd.read_csv('../data/processed/student_processed.csv')
print(f'Dataset shape: {df.shape}')

X = df.drop(columns=['performance'])
y = df['performance']

print(f'Features shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nTarget distribution:\n{y.value_counts()}')

## 3.2 Train / Validation / Test Split
Split: 70% Training, 15% Validation, 15% Test

In [ ]:
RANDOM_STATE = 42

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Second split: 50% of temp = 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f'Training set:   {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Test set:       {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')

print(f'\nTarget distribution in each set:')
for name, data in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    print(f'  {name}: {dict(data.value_counts())}')

In [ ]:
# Save split data for reproducibility
train_data = pd.concat([X_train, y_train], axis=1)
val_data = pd.concat([X_val, y_val], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_data.to_csv('../data/processed/train.csv', index=False)
val_data.to_csv('../data/processed/validation.csv', index=False)
test_data.to_csv('../data/processed/test.csv', index=False)

print('Split data saved to data/processed/')

## 3.3 Model Training
### Model 1: Logistic Regression

In [ ]:
lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
lr_model.fit(X_train, y_train)

lr_train_pred = lr_model.predict(X_train)
lr_val_pred = lr_model.predict(X_val)

print('=== Logistic Regression ===')
print(f'Training Accuracy:   {accuracy_score(y_train, lr_train_pred):.4f}')
print(f'Validation Accuracy: {accuracy_score(y_val, lr_val_pred):.4f}')
print(f'\nValidation Classification Report:')
print(classification_report(y_val, lr_val_pred, target_names=['Low', 'High']))

### Model 2: Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)

dt_train_pred = dt_model.predict(X_train)
dt_val_pred = dt_model.predict(X_val)

print('=== Decision Tree ===')
print(f'Training Accuracy:   {accuracy_score(y_train, dt_train_pred):.4f}')
print(f'Validation Accuracy: {accuracy_score(y_val, dt_val_pred):.4f}')
print(f'\nValidation Classification Report:')
print(classification_report(y_val, dt_val_pred, target_names=['Low', 'High']))

### Model 3: Random Forest

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)

rf_train_pred = rf_model.predict(X_train)
rf_val_pred = rf_model.predict(X_val)

print('=== Random Forest ===')
print(f'Training Accuracy:   {accuracy_score(y_train, rf_train_pred):.4f}')
print(f'Validation Accuracy: {accuracy_score(y_val, rf_val_pred):.4f}')
print(f'\nValidation Classification Report:')
print(classification_report(y_val, rf_val_pred, target_names=['Low', 'High']))

## 3.4 Model Comparison

In [ ]:
models = {
    'Logistic Regression': (lr_model, lr_val_pred),
    'Decision Tree': (dt_model, dt_val_pred),
    'Random Forest': (rf_model, rf_val_pred)
}

results = []
for name, (model, val_pred) in models.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_val, val_pred),
        'Precision': precision_score(y_val, val_pred),
        'Recall': recall_score(y_val, val_pred),
        'F1-Score': f1_score(y_val, val_pred),
        'AUC-ROC': roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
    })

results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')
print('=== Model Comparison (Validation Set) ===')
results_df.round(4)

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))
results_df.plot(kind='bar', ax=ax, colormap='viridis', edgecolor='black', alpha=0.8)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison (Validation Set)')
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
plt.xticks(rotation=0)

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, (model, val_pred)) in enumerate(models.items()):
    cm = confusion_matrix(y_val, val_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
    axes[i].set_title(f'{name}\nAccuracy: {accuracy_score(y_val, val_pred):.3f}')
    axes[i].set_ylabel('Actual')
    axes[i].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices (Validation Set)', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.6 ROC Curves

In [ ]:
plt.figure(figsize=(10, 7))

colors = ['#3498db', '#e74c3c', '#2ecc71']
for (name, (model, _)), color in zip(models.items(), colors):
    y_proba = model.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    auc = roc_auc_score(y_val, y_proba)
    plt.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../data/processed/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.7 Hyperparameter Tuning — Random Forest

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\nBest Parameters: {grid_search.best_params_}')
print(f'Best CV F1-Score: {grid_search.best_score_:.4f}')

In [ ]:
best_rf = grid_search.best_estimator_
best_rf_val_pred = best_rf.predict(X_val)

print('=== Tuned Random Forest (Validation Set) ===')
print(f'Accuracy:  {accuracy_score(y_val, best_rf_val_pred):.4f}')
print(f'Precision: {precision_score(y_val, best_rf_val_pred):.4f}')
print(f'Recall:    {recall_score(y_val, best_rf_val_pred):.4f}')
print(f'F1-Score:  {f1_score(y_val, best_rf_val_pred):.4f}')
print(f'AUC-ROC:   {roc_auc_score(y_val, best_rf.predict_proba(X_val)[:, 1]):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_val, best_rf_val_pred, target_names=['Low', 'High']))

## 3.8 Cross-Validation

In [ ]:
print('=== 5-Fold Cross-Validation Scores ===')
for name, model in [('Logistic Regression', lr_model), 
                     ('Decision Tree', dt_model), 
                     ('Random Forest (Tuned)', best_rf)]:
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    print(f'{name:30s} F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

## 3.9 Feature Importance (Random Forest)

In [ ]:
feature_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
top_n = 20
sns.barplot(x='importance', y='feature', data=feature_imp.head(top_n), 
            palette='viridis', edgecolor='black')
plt.title(f'Top {top_n} Feature Importances (Tuned Random Forest)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('../data/processed/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 Most Important Features:')
print(feature_imp.head(10).to_string(index=False))

## 3.10 Final Evaluation on Test Set

In [ ]:
test_pred = best_rf.predict(X_test)
test_proba = best_rf.predict_proba(X_test)[:, 1]

print('=== Final Test Set Evaluation (Tuned Random Forest) ===')
print(f'Accuracy:  {accuracy_score(y_test, test_pred):.4f}')
print(f'Precision: {precision_score(y_test, test_pred):.4f}')
print(f'Recall:    {recall_score(y_test, test_pred):.4f}')
print(f'F1-Score:  {f1_score(y_test, test_pred):.4f}')
print(f'AUC-ROC:   {roc_auc_score(y_test, test_proba):.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, test_pred, target_names=['Low', 'High']))

# Confusion Matrix
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'High'], yticklabels=['Low', 'High'])
plt.title('Test Set — Confusion Matrix (Tuned Random Forest)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('../data/processed/test_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.11 Save Best Model

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

# Save all models
joblib.dump(lr_model, '../models/logistic_regression.joblib')
joblib.dump(dt_model, '../models/decision_tree.joblib')
joblib.dump(best_rf, '../models/random_forest_best.joblib')

# Save model comparison results
results_df.to_csv('../data/processed/model_comparison_results.csv')

print('Models saved to ../models/')
print('Best model: random_forest_best.joblib')
print(f'Best parameters: {grid_search.best_params_}')

## Summary
- Trained 3 models: Logistic Regression, Decision Tree, Random Forest
- Random Forest performed best after hyperparameter tuning
- GridSearchCV used for hyperparameter optimization
- Cross-validation confirms model stability
- Key features: failures, absences, higher education aspiration, study time, parental education
- Best model saved for deployment